In [1]:
import os
import json
import pandas as pd
from utils import PATH
import re
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
capabilities_list = ['reasoning', 'comprehension', 'instruction following', 'agentic', 'knowledge retrieval', 'coding', 'multilingual']

/home/hshiah/anaconda3/envs/deepeval/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

def check_capabilities(attributes):
    capabilities_list = ['reasoning', 'comprehension', 'instruction following', 'agentic', 'knowledge retrieval', 'coding', 'multilingual']
    attributes = attributes.lower()
    index = []
    temp = {}
    for capability in capabilities_list:
        if attributes.find(capability) != -1:
            index.append(attributes.find(capability))
            temp[attributes.find(capability)] = capability
    output = [temp[index] for index in sorted(index)]
    return output
def remove_special_characters(text):
    # remove characters except letters, numbers, and spaces
    return re.sub(r'[^a-zA-Z\s]', '', text)
def check_knowledge(attributes):
    main_knowledge = []
    sub_knowledge = {}
    num_of_main = 1
    while True:
        if attributes.find(f'{num_of_main}. ') != -1:
            num_of_sub = 1
            main_knowledge_temp = attributes.split(f'{num_of_main}.')[1].split('\n')[0].strip()
            main_knowledge.append(remove_special_characters(main_knowledge_temp))
            sub_knowledge[main_knowledge_temp] = []
            while True:
                if attributes.find(f'{num_of_main}.{num_of_sub} ') != -1:
                    sub_knowledge[main_knowledge_temp].append(remove_special_characters(attributes.split(f'{num_of_main}.{num_of_sub} ')[1].split('\n')[0].strip()))
                    num_of_sub += 1
                else:
                    break
            num_of_main += 1
        else:
            break
    return main_knowledge, sub_knowledge
def compute_cos_sim(input: list[str]):
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    embeddings = model.encode(input)
    return cos_sim(embeddings[0], embeddings[1])
test = "Query: \"{A construction worker hit a solid steel wall with a sledgehammer, but the wall did not move. Part of the energy exerted by the worker\nA. was destroyed by the steel wall.\nB. was converted into heat.\nC. disappeared into the atmosphere.\nD. was stopped by the hard surface.\n\n\nOnly output the correct choice such as 'A', 'B', 'C' without any explanation. Full answer not needed. \nAnswer:}\"\n\nKnowledge: {\n1. Physics\n   1.1 Energy Conservation\n   1.2 Energy Transformation\n   1.3 Heat Generation\n\n2. Material"
check_knowledge(test)

(['Physics', 'Material'],
 {'Physics': ['Energy Conservation',
   'Energy Transformation',
   'Heat Generation'],
  'Material': []})

In [ ]:
capabilities_dict = {
    'reasoning': 0,
    'comprehension': 0,
    'instruction following': 0,
    'agentic': 0,
    'knowledge retrieval': 0,
    'coding': 0,
    # 'in-context learning': 0,
    'multilingual': 0
}
index_path = {'/home/hshiah/LLM_index/data/Cleared/ARC/data_capabilities_knowledge.jsonl':'ARC', 
              '/home/hshiah/LLM_index/data/Cleared/BIGBench/data_capabilities_knowledge.jsonl':'BIGBench',
                '/home/hshiah/LLM_index/data/Cleared/BigCodeBench/data_capabilities_knowledge.jsonl':'BigCodeBench',
                '/home/hshiah/LLM_index/data/Cleared/FinQA/data_capabilities_knowledge.jsonl':'FinQA',
                '/home/hshiah/LLM_index/data/Cleared/Flores/data_capabilities_knowledge.jsonl':'Flores',
                '/home/hshiah/LLM_index/data/Cleared/GSM8K/data_capabilities_knowledge.jsonl':'GSM8K',
                '/home/hshiah/LLM_index/data/Cleared/HiToM/data_capabilities_knowledge.jsonl':'HiToM',
                '/home/hshiah/LLM_index/data/Cleared/LegalBench/data_capabilities_knowledge.jsonl':'LegalBench',
                '/home/hshiah/LLM_index/data/Cleared/MATH-500/data_capabilities_knowledge.jsonl':'MATH',
                '/home/hshiah/LLM_index/data/Cleared/MedQA/data_capabilities_knowledge.jsonl':'MedQA',
                '/home/hshiah/LLM_index/data/Cleared/MMLU/data_capabilities_knowledge.jsonl':'MMLU',
                '/home/hshiah/LLM_index/data/Cleared/MMMLU/data_capabilities_knowledge.jsonl':'MMMLU',
                '/home/hshiah/LLM_index/data/Cleared/NaturalPlan/calendar_scheduling_capabilities_knowledge.jsonl':'NaturalPlanCalendarScheduling',
                '/home/hshiah/LLM_index/data/Cleared/NaturalPlan/meeting_planning_capabilities_knowledge.jsonl':'NaturalPlanMeetingPlanning',
                '/home/hshiah/LLM_index/data/Cleared/NaturalPlan/trip_planning_capabilities_knowledge.jsonl':'NaturalPlanTripPlanning',
                '/home/hshiah/LLM_index/data/Cleared/PlanBench/data_capabilities_knowledge.jsonl':'PlanBench',
                '/home/hshiah/LLM_index/data/Cleared/PubMedQA/data_capabilities_knowledge.jsonl':'PubMedQA',
                '/home/hshiah/LLM_index/data/Cleared/RACE/data_capabilities_knowledge.jsonl':'RACE',
                '/home/hshiah/LLM_index/data/Cleared/RuleTaker/data_capabilities_knowledge.jsonl':'RuleTaker',
                '/home/hshiah/LLM_index/data/Cleared/ScienceQA/data_capabilities_knowledge.jsonl':'ScienceQA',
                '/home/hshiah/LLM_index/data/Cleared/SciTLDR/data_capabilities_knowledge.jsonl':'SciTLDR',
                '/home/hshiah/LLM_index/data/Cleared/XSum/data_capabilities_knowledge.jsonl':'XSum'}

test_path = {'/home/hshiah/LLM_index/data/Cleared/MMLUPro/data_capabilities_knowledge.jsonl': 'MMLUPro',
             '/home/hshiah/LLM_index/data/Cleared/BigGenBench/data_capabilities_knowledge.jsonl': 'BigGenBench',
             '/home/hshiah/LLM_index/data/Cleared/GPQA/data_capabilities_knowledge.jsonl':'GPQA',
             '/home/hshiah/LLM_index/data/Cleared/LiveBench/coding/data.jsonl': 'coding',
             '/home/hshiah/LLM_index/data/Cleared/LiveBench/data_analysis/data.jsonl': 'data_analysis',
            '/home/hshiah/LLM_index/data/Cleared/LiveBench/language/data.jsonl': 'language',
            '/home/hshiah/LLM_index/data/Cleared/LiveBench/math/data.jsonl': 'math',
            '/home/hshiah/LLM_index/data/Cleared/LiveBench/reasoning/data.jsonl': 'reasoning',
             '/home/hshiah/LLM_index/data/Cleared/LiveBench/instruction_following/data.jsonl': 'instruction_following'}

replaced_path = "_attributes_qwen-2.5-7b-instruct.jsonl"
# replaced_path = "_attributes_llama-3.1-8b-instruct.jsonl"
# replaced_path = "_attributes_gemma-3-12b-it.jsonl"
for path,task in index_path.items():
    if 'LiveBench' not in path:
        path = path.replace("_capabilities_knowledge.jsonl",replaced_path)
    else:
        path = path.replace('.jsonl', replaced_path)
    df = pd.read_json(path, lines=True)
    output_list = []
    for i in range(len(df)):
        capabilities = check_capabilities(df['capabilities'][i])
        output_list.append(capabilities)
        if len(capabilities) == 0:
            # print(df['capabilities'][i])
            print(task, i)
        for capability in capabilities:
            capabilities_dict[capability] += 1
            break
    df['capabilities_list'] = output_list
    df.to_json(path, orient='records', lines=True, index=False)

for path,task in test_path.items():
    if 'LiveBench' not in path:
        path = path.replace("_capabilities_knowledge.jsonl",replaced_path)
    else:
        path = path.replace('.jsonl', replaced_path)
    df = pd.read_json(path, lines=True)
    output_list = []
    for i in range(len(df)):
        capabilities = check_capabilities(df['capabilities'][i])
        output_list.append(capabilities)
        if len(capabilities) == 0:
            # print(df['capabilities'][i])
            print(task, i)
        for capability in capabilities:
            capabilities_dict[capability] += 1
            break
    df['capabilities_list'] = output_list
    df.to_json(path, orient='records', lines=True, index=False)
print(capabilities_dict)

/tmp/ipykernel_3753475/2736889265.py:59: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df = pd.read_json(path, lines=True)
/tmp/ipykernel_3753475/2736889265.py:59: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df = pd.read_json(path, lines=True)
/tmp/ipykernel_3753475/2736889265.py:59: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. T

{'reasoning': 19332, 'comprehension': 4321, 'instruction following': 1258, 'agentic': 133, 'knowledge retrieval': 1187, 'coding': 315, 'multilingual': 173}


In [4]:
# make it a set
# def merge_similar_phrases(knowledge_dict, similarity_matrix, threshold=0.7):
#     """
#     Merge similar phrases based on cosine similarity threshold
#     Args:
#         knowledge_dict: Dictionary with phrases as keys and their frequencies as values
#         similarity_matrix: Matrix of cosine similarities between phrases
#         threshold: Similarity threshold for merging (default: 0.7)
#     Returns:
#         Dictionary with merged phrases and their combined frequencies
#     """
#     # Convert dictionary to list of phrases for indexing
#     knowledge_set = list(knowledge_dict.keys())
    
#     # Create a dictionary to store merged phrases
#     merged_phrases = defaultdict(list)
#     processed_indices = set()
    
#     # Find similar pairs
#     for i in range(len(knowledge_set)):
#         if i in processed_indices:
#             continue
            
#         current_phrase = knowledge_set[i]
#         similar_phrases = [current_phrase]
#         current_freq = knowledge_dict[current_phrase]
        
#         # Find all similar phrases to current phrase
#         for j in range(i + 1, len(knowledge_set)):
#             if j in processed_indices:
#                 continue
                
#             if similarity_matrix[i, j] > threshold:
#                 similar_phrases.append(knowledge_set[j])
#                 processed_indices.add(j)
        
#         if len(similar_phrases) > 1:
#             # Find the phrase with highest frequency to use as key
#             key_phrase = max(similar_phrases, key=lambda x: knowledge_dict[x])
#             merged_phrases[key_phrase].extend(similar_phrases)
#             processed_indices.add(i)
#         else:
#             # If no similar phrases found, keep the original
#             merged_phrases[current_phrase].append(current_phrase)
#             processed_indices.add(i)
    
#     # Create final dictionary with merged frequencies
#     final_merged_dict = {}
#     for key_phrase, similar_phrases in merged_phrases.items():
#         # Sum up frequencies of all similar phrases
#         total_freq = sum(knowledge_dict[phrase] for phrase in similar_phrases)
#         final_merged_dict[key_phrase] = {
#             'frequency': total_freq,
#             'similar_phrases': similar_phrases
#         }
    
#     return final_merged_dict

def merge_similar_phrases(knowledge_dict, similarity_matrix, threshold=0.6, min_frequency=10):
    """
    Merge similar phrases based on cosine similarity threshold
    Args:
        knowledge_dict: Dictionary with phrases as keys and their frequencies as values
        similarity_matrix: Matrix of cosine similarities between phrases
        threshold: Similarity threshold for merging (default: 0.7)
        min_frequency: Minimum frequency to keep a phrase (default: 10)
    Returns:
        Dictionary with merged phrases and their combined frequencies
    """
    # Convert dictionary to list of phrases for indexing
    knowledge_set = list(knowledge_dict.keys())
    
    # Create a dictionary to store merged phrases
    merged_phrases = defaultdict(list)
    processed_indices = set()
    
    # Find similar pairs
    for i in range(len(knowledge_set)):
        if i in processed_indices:
            continue
            
        current_phrase = knowledge_set[i]
        similar_phrases = [current_phrase]
        current_freq = knowledge_dict[current_phrase]
        
        # Find all similar phrases to current phrase
        for j in range(i + 1, len(knowledge_set)):
            if j in processed_indices:
                continue
                
            if similarity_matrix[i, j] > threshold:
                similar_phrases.append(knowledge_set[j])
                processed_indices.add(j)
        
        if len(similar_phrases) > 1:
            # Find the phrase with highest frequency to use as key
            key_phrase = max(similar_phrases, key=lambda x: knowledge_dict[x])
            merged_phrases[key_phrase].extend(similar_phrases)
            processed_indices.add(i)
        else:
            # If no similar phrases found, keep the original
            merged_phrases[current_phrase].append(current_phrase)
            processed_indices.add(i)
    
    # Create final dictionary with merged frequencies
    final_merged_dict = {}
    other_phrases = []
    other_frequency = 0
    
    for key_phrase, similar_phrases in merged_phrases.items():
        # Sum up frequencies of all similar phrases
        total_freq = sum(knowledge_dict[phrase] for phrase in similar_phrases)
        
        if total_freq >= min_frequency:
            final_merged_dict[key_phrase] = {
                'frequency': total_freq,
                'similar_phrases': similar_phrases
            }
        else:
            # Add to other category
            other_phrases.extend(similar_phrases)
            other_frequency += total_freq
    
    # Add "other" category if there are low-frequency phrases
    if other_phrases:
        final_merged_dict["other"] = {
            'frequency': other_frequency,
            'similar_phrases': other_phrases
        }
    
    return final_merged_dict
# def substitute_phrases(phrases_list, merged_phrases):
#     """
#     Substitute phrases in a list using the merged phrases dictionary
#     Args:
#         phrases_list: List of phrases to be substituted
#         merged_phrases: Dictionary of merged phrases from merge_similar_phrases
#     Returns:
#         List of substituted phrases
#     """
#     # Create a mapping dictionary for quick lookup
#     phrase_mapping = {}
#     for key_phrase, data in merged_phrases.items():
#         for phrase in data['similar_phrases']:
#             phrase_mapping[phrase] = key_phrase
    
#     # Substitute phrases
#     substituted_phrases = []
#     for phrase in phrases_list:
#         # If phrase exists in mapping, use the key phrase, otherwise keep original
#         substituted_phrase = phrase_mapping.get(phrase, phrase)
#         substituted_phrases.append(substituted_phrase)
    
#     return substituted_phrases
def substitute_phrases(phrases_list, merged_phrases):
    """
    Substitute phrases in a list using the merged phrases dictionary and remove duplicates
    Args:
        phrases_list: List of phrases to be substituted
        merged_phrases: Dictionary of merged phrases from merge_similar_phrases
    Returns:
        List of substituted phrases with duplicates removed
    """
    # Create a mapping dictionary for quick lookup
    phrase_mapping = {}
    for key_phrase, data in merged_phrases.items():
        for phrase in data['similar_phrases']:
            phrase_mapping[phrase] = key_phrase
    
    # Substitute phrases and remove duplicates
    substituted_phrases = []
    seen_phrases = set()  # To track unique phrases
    
    for phrase in phrases_list:
        # If phrase exists in mapping, use the key phrase, otherwise keep original
        substituted_phrase = phrase_mapping.get(phrase, "other")
        
        # Only add if we haven't seen this phrase before
        if substituted_phrase not in seen_phrases:
            substituted_phrases.append(substituted_phrase)
            seen_phrases.add(substituted_phrase)
    
    return substituted_phrases

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
knowledge_set = []
knowledge_dict = {}
for path,task in index_path.items():
    if 'LiveBench' not in path:
        path = path.replace("_capabilities_knowledge.jsonl",replaced_path)
    else:
        path = path.replace('.jsonl', replaced_path)
    df = pd.read_json(path, lines=True)
    for i in range(len(df)):
            # main_knowledge, sub_knowledge = check_knowledge(df['knowledge'][i])
            main_knowledge, sub_knowledge = check_knowledge(df['capabilities'][i].split('Knowledge:')[-1])
            # print(main_knowledge)
            if len(main_knowledge) == 0:
                print(task, i)
            
            for knowledge in main_knowledge:
                if knowledge not in knowledge_set:
                    knowledge_set.append(knowledge)
                    knowledge_dict[knowledge] = 1
                else:
                    knowledge_dict[knowledge] += 1
embeddings = model.encode(knowledge_set, batch_size=128, convert_to_tensor=False)
print(embeddings.shape)
similarity_matrix = cosine_similarity(embeddings)
mapping = merge_similar_phrases(knowledge_dict, similarity_matrix, 0.7)
for path,task in index_path.items():
    if 'LiveBench' not in path:
        path = path.replace("_capabilities_knowledge.jsonl",replaced_path)
    else:
        path = path.replace('.jsonl', replaced_path)
    df = pd.read_json(path, lines=True)
    main_knowledge_list = []
    for i in range(len(df)):
        main_knowledge, sub_knowledge = check_knowledge(df['capabilities'][i].split('Knowledge:')[-1])
        main_knowledge = substitute_phrases(main_knowledge, mapping)
        if 'other' in main_knowledge:
            # main_knowledge.remove('other')
            pass
        main_knowledge_list.append(main_knowledge)
    df['main_knowledge'] = main_knowledge_list
    df.to_json(path, orient='records', lines=True, index=False)
print(mapping)
count_unhit_knowledge = 0
total_knowledge = 0
for path,task in test_path.items():
    if 'LiveBench' not in path:
        path = path.replace("_capabilities_knowledge.jsonl",replaced_path)
    else:
        path = path.replace('.jsonl', replaced_path)
    df = pd.read_json(path, lines=True)
    main_knowledge_list = []
    for i in range(len(df)):
        main_knowledge, sub_knowledge = check_knowledge(df['capabilities'][i].split('Knowledge:')[-1])
        main_knowledge = substitute_phrases(main_knowledge, mapping)
        total_knowledge += len(main_knowledge)
        if 'other' in main_knowledge:
            count_unhit_knowledge += 1
            # main_knowledge.remove('other')
        main_knowledge_list.append(main_knowledge)
    df['main_knowledge'] = main_knowledge_list
    df.to_json(path, orient='records', lines=True, index=False)
print(count_unhit_knowledge)
print(total_knowledge)
# print top 10 frequent knowledge
knowledge_dict = sorted(knowledge_dict.items(), key=lambda x: x[1], reverse=True)
print(knowledge_dict)



/tmp/ipykernel_3753475/3890545465.py:193: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df = pd.read_json(path, lines=True)
/tmp/ipykernel_3753475/3890545465.py:193: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df = pd.read_json(path, lines=True)
/tmp/ipykernel_3753475/3890545465.py:193: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'

(3895, 384)


/tmp/ipykernel_3753475/3890545465.py:216: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df = pd.read_json(path, lines=True)
/tmp/ipykernel_3753475/3890545465.py:216: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df = pd.read_json(path, lines=True)
/tmp/ipykernel_3753475/3890545465.py:216: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'

{'Biology': {'frequency': 667, 'similar_phrases': ['Biology', 'Animal Biology', 'Molecular Biology', 'Cellular Biology', 'Evolutionary Biology', 'Genetics  Biology', 'Species and Biology', 'Human Biology', 'Biological Sciences', 'Biology  Medicine', 'Biological']}, 'Zoology': {'frequency': 137, 'similar_phrases': ['Zoology', 'Zoology  Data Analysis', 'Biology  Zoology  Data Analysis', 'Zoological']}, 'Immunology': {'frequency': 67, 'similar_phrases': ['Immunology', 'Immunohistochemistry', 'Immunoassay', 'Immunocytochemistry']}, 'Anatomy': {'frequency': 252, 'similar_phrases': ['Anatomy', 'Comparative Anatomy', 'Human Anatomy', 'Anatomy and Physiology', 'Medicine  Anatomy', 'Physiology  Anatomy', 'Anatomy  Medicine', 'Anatomy  Physiology']}, 'Ecology': {'frequency': 126, 'similar_phrases': ['Ecology', 'Home Ecology', 'Nature and Environment', 'Nature and wildlife', 'Natural Environment']}, 'Chemistry': {'frequency': 264, 'similar_phrases': ['Chemistry', 'Physics  Chemistry', 'Physical C